# S8.2 — AI-Generated Paper Summary

**Spec**: `specs/spec-S8.2-paper-summary/spec.md`

Verifies:
- `SummarizerService` generates structured summaries (objective, method, key_findings, contribution, limitations)
- Content extraction handles full papers and abstract-only cases
- `POST /api/v1/papers/{paper_id}/summary` endpoint works correctly
- Edge cases: paper not found, insufficient content, LLM failure

In [1]:
import sys
sys.path.insert(0, "../..")

import asyncio
import nest_asyncio
nest_asyncio.apply()

import uuid
from unittest.mock import AsyncMock, MagicMock

print("✓ Imports and path setup complete")

✓ Imports and path setup complete


## 1. Schema Validation — PaperSummary & SummaryResponse

In [2]:
from src.schemas.api.analysis import PaperSummary, SummaryResponse

paper_id = uuid.uuid4()

summary = PaperSummary(
    objective="Propose a new architecture based solely on attention.",
    method="Multi-head self-attention with positional encoding.",
    key_findings=[
        "Achieves 28.4 BLEU on WMT 2014 EN-DE.",
        "Training is significantly faster than recurrent models.",
    ],
    contribution="Introduces the Transformer, based entirely on attention.",
    limitations="Quadratic complexity with sequence length.",
)

response = SummaryResponse(
    paper_id=paper_id,
    title="Attention Is All You Need",
    summary=summary,
    model="gemini-3-flash",
    provider="gemini",
    latency_ms=1200.0,
    warnings=[],
)

assert response.summary.objective
assert len(response.summary.key_findings) == 2
assert response.model == "gemini-3-flash"
assert response.paper_id == paper_id

print(f"PaperSummary fields: {list(summary.model_fields.keys())}")
print(f"SummaryResponse fields: {list(response.model_fields.keys())}")
print(f"key_findings count: {len(response.summary.key_findings)}")
print("\n✓ Schema validation passed")

PaperSummary fields: ['objective', 'method', 'key_findings', 'contribution', 'limitations']
SummaryResponse fields: ['paper_id', 'title', 'summary', 'model', 'provider', 'latency_ms', 'warnings']
key_findings count: 2

✓ Schema validation passed


/var/folders/09/369xpjkj0bx605z54hlxdxv80000gn/T/ipykernel_18136/2839164950.py:31: PydanticDeprecatedSince211: Accessing the 'model_fields' attribute on the instance is deprecated. Instead, you should access this attribute from the model class. Deprecated in Pydantic V2.11 to be removed in V3.0.
  print(f"PaperSummary fields: {list(summary.model_fields.keys())}")
/var/folders/09/369xpjkj0bx605z54hlxdxv80000gn/T/ipykernel_18136/2839164950.py:32: PydanticDeprecatedSince211: Accessing the 'model_fields' attribute on the instance is deprecated. Instead, you should access this attribute from the model class. Deprecated in Pydantic V2.11 to be removed in V3.0.
  print(f"SummaryResponse fields: {list(response.model_fields.keys())}")


## 2. Content Extraction — Full Paper & Truncation

In [3]:
from src.services.analysis.summarizer import SummarizerService

service = SummarizerService()

# Mock a paper with sections
paper = MagicMock()
paper.title = "Attention Is All You Need"
paper.authors = ["Vaswani, A.", "Shazeer, N."]
paper.abstract = "We propose a new simple network architecture, the Transformer."
paper.sections = [
    {"title": "Introduction", "content": "Recurrent models are slow.", "level": 1},
    {"title": "Methodology", "content": "Multi-head attention with scaled dot-product.", "level": 1},
    {"title": "Results", "content": "28.4 BLEU on WMT 2014 EN-DE.", "level": 1},
    {"title": "Conclusion", "content": "First model based entirely on attention.", "level": 1},
]
paper.pdf_content = None

content = service.extract_content(paper)

assert "Attention Is All You Need" in content
assert "Transformer" in content
assert "Multi-head attention" in content
assert "28.4 BLEU" in content

print(f"Extracted content length: {len(content)} chars, {len(content.split())} words")
print(f"First 200 chars:\n{content[:200]}")

# Test truncation with very long content
long_paper = MagicMock()
long_paper.title = "Long Paper"
long_paper.authors = []
long_paper.abstract = "Short abstract."
long_paper.sections = [{"title": "Body", "content": "word " * 5000, "level": 1}]
long_paper.pdf_content = None

long_content = service.extract_content(long_paper)
word_count = len(long_content.split())
assert word_count <= 4500, f"Content too long: {word_count} words"

print(f"\nLong paper truncated to: {word_count} words (limit ~4000)")
print("\n✓ Content extraction and truncation passed")

Extracted content length: 343 chars, 50 words
First 200 chars:
Title: Attention Is All You Need
Authors: Vaswani, A., Shazeer, N.

Abstract:
We propose a new simple network architecture, the Transformer.

## Introduction
Recurrent models are slow.

## Methodology

Long paper truncated to: 4000 words (limit ~4000)

✓ Content extraction and truncation passed


## 3. Summary Generation (Mocked LLM)

In [4]:
STRUCTURED_LLM_OUTPUT = """{
    "objective": "Propose a new architecture based solely on attention.",
    "method": "Multi-head self-attention with positional encoding.",
    "key_findings": [
        "Achieves 28.4 BLEU on WMT 2014 EN-DE.",
        "Training is significantly faster than recurrent models."
    ],
    "contribution": "Introduces the Transformer, based entirely on attention.",
    "limitations": "Quadratic complexity with sequence length."
}"""

# Mock LLM response
mock_llm_response = MagicMock()
mock_llm_response.text = STRUCTURED_LLM_OUTPUT
mock_llm_response.model = "gemini-3-flash"
mock_llm_response.provider = "gemini"
mock_llm_response.usage = MagicMock()
mock_llm_response.usage.latency_ms = 1200.0
mock_llm_response.usage.total_tokens = 500

# Mock paper
mock_paper = MagicMock()
mock_paper.id = uuid.uuid4()
mock_paper.title = "Attention Is All You Need"
mock_paper.authors = ["Vaswani, A."]
mock_paper.abstract = "We propose the Transformer."
mock_paper.sections = [
    {"title": "Introduction", "content": "Recurrent models are slow.", "level": 1},
]
mock_paper.pdf_content = None
mock_paper.arxiv_id = "1706.03762"

# Mock repo and LLM
mock_repo = AsyncMock()
mock_repo.get_by_id = AsyncMock(return_value=mock_paper)
mock_session = AsyncMock()
mock_llm = AsyncMock()
mock_llm.generate = AsyncMock(return_value=mock_llm_response)

service = SummarizerService()
result = asyncio.get_event_loop().run_until_complete(
    service.summarize(
        paper_id=mock_paper.id,
        paper_repo=mock_repo,
        session=mock_session,
        llm_provider=mock_llm,
    )
)

assert result.summary.objective
assert result.summary.method
assert len(result.summary.key_findings) >= 1
assert result.summary.contribution
assert result.summary.limitations
assert result.paper_id == mock_paper.id
assert result.model == "gemini-3-flash"

print(f"Objective: {result.summary.objective}")
print(f"Method: {result.summary.method}")
print(f"Key findings ({len(result.summary.key_findings)}):")
for i, f in enumerate(result.summary.key_findings, 1):
    print(f"  {i}. {f}")
print(f"Contribution: {result.summary.contribution}")
print(f"Limitations: {result.summary.limitations}")
print(f"Model: {result.model} ({result.provider})")
print(f"Latency: {result.latency_ms}ms")

mock_llm.generate.assert_called_once()
print("\n✓ Summary generation with mocked LLM passed")

Objective: Propose a new architecture based solely on attention.
Method: Multi-head self-attention with positional encoding.
Key findings (2):
  1. Achieves 28.4 BLEU on WMT 2014 EN-DE.
  2. Training is significantly faster than recurrent models.
Contribution: Introduces the Transformer, based entirely on attention.
Limitations: Quadratic complexity with sequence length.
Model: gemini-3-flash (gemini)
Latency: 1200.0ms

✓ Summary generation with mocked LLM passed


## 4. Edge Cases — Not Found, Insufficient Content, LLM Failure

In [5]:
from src.exceptions import InsufficientContentError, LLMServiceError, PaperNotFoundError

service = SummarizerService()

# --- Paper not found ---
mock_repo = AsyncMock()
mock_repo.get_by_id = AsyncMock(return_value=None)
mock_session = AsyncMock()
mock_llm = AsyncMock()

try:
    asyncio.get_event_loop().run_until_complete(
        service.summarize(
            paper_id=uuid.uuid4(),
            paper_repo=mock_repo,
            session=mock_session,
            llm_provider=mock_llm,
        )
    )
    assert False, "Should have raised PaperNotFoundError"
except PaperNotFoundError:
    print("✓ PaperNotFoundError raised for missing paper")

# --- Insufficient content ---
empty_paper = MagicMock()
empty_paper.id = uuid.uuid4()
empty_paper.title = "Empty Paper"
empty_paper.abstract = ""
empty_paper.sections = None
empty_paper.pdf_content = None
empty_paper.authors = []

mock_repo.get_by_id = AsyncMock(return_value=empty_paper)

try:
    asyncio.get_event_loop().run_until_complete(
        service.summarize(
            paper_id=empty_paper.id,
            paper_repo=mock_repo,
            session=mock_session,
            llm_provider=mock_llm,
        )
    )
    assert False, "Should have raised InsufficientContentError"
except InsufficientContentError:
    print("✓ InsufficientContentError raised for empty paper")

# --- LLM failure ---
paper_with_content = MagicMock()
paper_with_content.id = uuid.uuid4()
paper_with_content.title = "Good Paper"
paper_with_content.abstract = "Has content."
paper_with_content.sections = [{"title": "Intro", "content": "Some text.", "level": 1}]
paper_with_content.pdf_content = None
paper_with_content.authors = ["Author"]

mock_repo.get_by_id = AsyncMock(return_value=paper_with_content)
mock_llm.generate = AsyncMock(side_effect=LLMServiceError("LLM unavailable"))

try:
    asyncio.get_event_loop().run_until_complete(
        service.summarize(
            paper_id=paper_with_content.id,
            paper_repo=mock_repo,
            session=mock_session,
            llm_provider=mock_llm,
        )
    )
    assert False, "Should have raised LLMServiceError"
except LLMServiceError:
    print("✓ LLMServiceError raised for LLM failure")

print("\n✓ All edge cases passed")

✓ PaperNotFoundError raised for missing paper
✓ InsufficientContentError raised for empty paper
✓ LLMServiceError raised for LLM failure

✓ All edge cases passed


## 5. Endpoint Test — POST /api/v1/papers/{paper_id}/summary

In [6]:
from fastapi import FastAPI
from httpx import ASGITransport, AsyncClient
from src.dependency import get_db_session, get_llm_provider, get_paper_repository
from src.routers.analysis import router

# Build minimal app
app = FastAPI()
app.include_router(router, prefix="/api/v1")

# Mock paper
test_paper_id = uuid.uuid4()
mock_paper = MagicMock()
mock_paper.id = test_paper_id
mock_paper.title = "Attention Is All You Need"
mock_paper.authors = ["Vaswani, A."]
mock_paper.abstract = "We propose the Transformer."
mock_paper.sections = [
    {"title": "Introduction", "content": "Recurrent models are slow.", "level": 1},
]
mock_paper.pdf_content = None

# Mock deps
mock_repo = AsyncMock()
mock_repo.get_by_id = AsyncMock(return_value=mock_paper)
mock_session = AsyncMock()

mock_llm_response = MagicMock()
mock_llm_response.text = STRUCTURED_LLM_OUTPUT
mock_llm_response.model = "gemini-3-flash"
mock_llm_response.provider = "gemini"
mock_llm_response.usage = MagicMock()
mock_llm_response.usage.latency_ms = 950.0
mock_llm = AsyncMock()
mock_llm.generate = AsyncMock(return_value=mock_llm_response)

app.dependency_overrides[get_db_session] = lambda: mock_session
app.dependency_overrides[get_paper_repository] = lambda: mock_repo
app.dependency_overrides[get_llm_provider] = lambda: mock_llm

async def _test_endpoint():
    transport = ASGITransport(app=app)
    async with AsyncClient(transport=transport, base_url="http://test") as client:
        # 200 success
        resp = await client.post(f"/api/v1/papers/{test_paper_id}/summary")
        assert resp.status_code == 200, f"Expected 200, got {resp.status_code}: {resp.text}"
        data = resp.json()
        assert "summary" in data
        assert data["summary"]["objective"]
        assert data["paper_id"] == str(test_paper_id)
        print(f"POST /api/v1/papers/{test_paper_id}/summary -> 200")
        print(f"  objective: {data['summary']['objective']}")

        # 404 not found
        mock_repo.get_by_id = AsyncMock(return_value=None)
        fake_id = uuid.uuid4()
        resp = await client.post(f"/api/v1/papers/{fake_id}/summary")
        assert resp.status_code == 404, f"Expected 404, got {resp.status_code}"
        print(f"POST /api/v1/papers/{fake_id}/summary -> 404")

asyncio.get_event_loop().run_until_complete(_test_endpoint())
app.dependency_overrides.clear()

print("\n✓ Endpoint tests passed")

POST /api/v1/papers/96e69a35-fdea-49c7-8625-1bb1dec98b81/summary -> 200
  objective: Propose a new architecture based solely on attention.
POST /api/v1/papers/7fe5c4e4-f1fa-487e-87ba-4e2fb5a7508b/summary -> 404

✓ Endpoint tests passed


## Summary

All S8.2 checks passed:
- `PaperSummary` and `SummaryResponse` Pydantic schemas validate correctly
- `SummarizerService.extract_content()` handles full papers and truncation
- `SummarizerService.summarize()` generates structured summaries via LLM
- Edge cases: `PaperNotFoundError`, `InsufficientContentError`, `LLMServiceError`
- `POST /api/v1/papers/{paper_id}/summary` returns 200/404 correctly